<a href="https://colab.research.google.com/github/monees007/ML-bace1-inhibitors/blob/master/Docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# sync to drive
!mkdir -p "/content/drive/MyDrive/BACE1_Project"
!rsync -av --progress /content/ "/content/drive/MyDrive/BACE1_Project/" --exclude "sample_data" --exclude ".config" --exclude "drive"

sending incremental file list
./
bace1_data_with_labels.csv
         28,177 100%    0.00kB/s    0:00:00 (xfr#1, to-chk=28/32)
candidates.smi
          8,990 100%    8.57MB/s    0:00:00 (xfr#2, to-chk=27/32)

sent 38,174 bytes  received 144 bytes  76,636.00 bytes/sec
total size is 24,427,837  speedup is 637.50


In [ ]:
!rm autodock-gpu
!wget https://github.com/ccsb-scripps/AutoDock-GPU/releases/download/v1.6/adgpu-v1.6_linux_x64_cuda12_128wi -O autodock-gpu
!chmod +x autodock-gpu
!ln -sf /usr/local/cuda/lib64/libcurand.so /usr/local/cuda/lib64/libcurand.so.10
!ldconfig

!./autodock-gpu

In [31]:
# 1. Download with --no-check-certificate to bypass the SSL error
!wget --no-check-certificate https://autodock.scripps.edu/wp-content/uploads/sites/56/2021/10/autodocksuite-4.2.6-x86_64Linux2.tar

# 2. Extract the tarball
!tar -xvf autodocksuite-4.2.6-x86_64Linux2.tar

# 3. Move the binary to /usr/bin so you can run it
!mv x86_64Linux2/autogrid4 /usr/bin/
!chmod +x /usr/bin/autogrid4

# 4. Clean up
!rm -rf x86_64Linux2 autodocksuite-4.2.6-x86_64Linux2.tar

# 5. Verify it works now
!autogrid4 --version
!sudo apt install openbabel


--2026-02-13 08:04:07--  https://autodock.scripps.edu/wp-content/uploads/sites/56/2021/10/autodocksuite-4.2.6-x86_64Linux2.tar
Resolving autodock.scripps.edu (autodock.scripps.edu)... 192.26.252.19
Connecting to autodock.scripps.edu (autodock.scripps.edu)|192.26.252.19|:443... connected.
	requested host name ‘autodock.scripps.edu’.
HTTP request sent, awaiting response... 200 OK
Length: 743424 (726K) [application/x-tar]
Saving to: ‘autodocksuite-4.2.6-x86_64Linux2.tar’

autodocksuite-4.2.6 100%[===================>] 726.00K  --.-KB/s    in 0.05s   

2026-02-13 08:04:07 (13.0 MB/s) - ‘autodocksuite-4.2.6-x86_64Linux2.tar’ saved [743424/743424]

x86_64Linux2/autodock4
x86_64Linux2/autogrid4
AutoGrid 4.2.6   
compilation options:
  Double-precision calculations (USE_DOUBLE):  yes
  Non-bond cutoff for internal energy calculation (NBC): 8.00
  Optimize internal energy scoring (USE_8A_NBCUTOFF):  yes
  Maximum number of receptor atom types (NUM_RECEPTOR_TYPES): 20
  Maximum number of atom ty

In [ ]:
receptor_file = "protein.pdbqt"
center_x = 10.261
center_y = -3.492
center_z = -0.127
box_size = 60      # 60 points (approx 22.5 Angstroms with 0.375 spacing)

gpf_content = f"""
npts {box_size} {box_size} {box_size}                        # num.grid points in xyz
gridfld {receptor_file[:-6]}.maps.fld                 # grid_data_file
spacing 0.375                             # spacing(A)
receptor_types A C HD N NA OA SA          # receptor atom types
ligand_types A C F NA OA N HD Br Cl I S P # ligand atom types
smooth 0.5                                # store minimum energy w/in rad(A)
map {receptor_file[:-6]}.A.map            # atom-specific affinity map
map {receptor_file[:-6]}.C.map            # atom-specific affinity map
map {receptor_file[:-6]}.F.map            # atom-specific affinity map
map {receptor_file[:-6]}.NA.map           # atom-specific affinity map
map {receptor_file[:-6]}.OA.map           # atom-specific affinity map
map {receptor_file[:-6]}.N.map            # atom-specific affinity map
map {receptor_file[:-6]}.HD.map           # atom-specific affinity map
map {receptor_file[:-6]}.Br.map           # atom-specific affinity map
map {receptor_file[:-6]}.Cl.map           # atom-specific affinity map
map {receptor_file[:-6]}.I.map            # atom-specific affinity map
map {receptor_file[:-6]}.S.map            # atom-specific affinity map
map {receptor_file[:-6]}.P.map            # atom-specific affinity map
elecmap {receptor_file[:-6]}.e.map        # electrostatic potential map
dsolvmap {receptor_file[:-6]}.d.map              # desolvation potential map
dielectric -0.1465                        # <0, AD4 distance-dep.diel;>0, constant
gridcenter {center_x} {center_y} {center_z}      # xyz-coordinates or auto
"""

with open("grid.gpf", "w") as f:
    f.write(gpf_content)
print("Generated grid.gpf successfully!")

Generated grid.gpf successfully!


In [ ]:
receptor_name = "protein" # Match your filename without .pdbqt

fld_content = f"""
label=AutoDock-GPU-Maps
#
variable 1 file={receptor_name}.A.map filetype=ascii skip=6
variable 2 file={receptor_name}.C.map filetype=ascii skip=6
variable 3 file={receptor_name}.F.map filetype=ascii skip=6
variable 4 file={receptor_name}.NA.map filetype=ascii skip=6
variable 5 file={receptor_name}.OA.map filetype=ascii skip=6
variable 6 file={receptor_name}.N.map filetype=ascii skip=6
variable 7 file={receptor_name}.HD.map filetype=ascii skip=6
variable 8 file={receptor_name}.Br.map filetype=ascii skip=6
variable 9 file={receptor_name}.Cl.map filetype=ascii skip=6
variable 10 file={receptor_name}.I.map filetype=ascii skip=6
variable 11 file={receptor_name}.S.map filetype=ascii skip=6
variable 12 file={receptor_name}.P.map filetype=ascii skip=6
variable 13 file={receptor_name}.e.map filetype=ascii skip=6
variable 14 file={receptor_name}.d.map filetype=ascii skip=6
"""

with open(f"{receptor_name}.maps.fld", "w") as f:
    f.write(fld_content)
print(f"Generated {receptor_name}.maps.fld")

Generated protein.maps.fld


In [ ]:
!autogrid4 -p grid.gpf -l grid.glg

In [ ]:
!./autodock-gpu --ffile protein.maps.fld --lfile ligand_ref.pdbqt --nrun 20

AutoDock-GPU version: v1.6

Running 1 docking calculation

Cuda device:                              Tesla T4
Available memory on device:               14807 MB (total: 14912 MB)

CUDA Setup time 0.283392s
(Thread 1 is setting up Job #1)

Running Job #1
    Using heuristics: (capped) number of evaluations set to 11867937
             This means this docking may not be able to converge. Increasing --heurmax may improve
             convergence but will also increase runtime.
             AutoStop will not stop before 99.67% (11829247) of the set number of evaluations.
    Local-search chosen method is: ADADELTA (ad)

Rest of Setup time 0.018007s

Executing docking runs, stopping automatically after either reaching 0.15 kcal/mol standard deviation of
the best molecules of the last 4 * 5 generations, 42000 generations, or 11867937 evaluations:

Generations |  Evaluations |     Threshold    |  Average energy of best 10%  | Samples | Best Inter + Intra
------------+--------------+----------

In [ ]:
import glob
import re

def get_best_score(dlg_filename):
    lowest_energy = 0.0
    with open(dlg_filename, 'r') as f:
        for line in f:
            # Look for lines like: "DOCKED: USER    Estimated Free Energy of Binding    =  -9.55 kcal/mol"
            if "Estimated Free Energy of Binding" in line:
                parts = line.split()
                # The energy is usually the index before "kcal/mol"
                energy = float(parts[parts.index("=") + 1])
                if energy < lowest_energy:
                    lowest_energy = energy
    return lowest_energy

# Find the file (AutoDock-GPU usually names it based on the ligand)
dlg_files = glob.glob("*.dlg")
if dlg_files:
    print(f"Found log file: {dlg_files[0]}")
    score = get_best_score(dlg_files[0])
    print(f"🏆 Best Docking Score: {score} kcal/mol")
else:
    print("❌ No DLG file found. Did the docking finish?")

In [33]:
!pip install chembl_webresource_client

from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np
import os

# 1. Setup ChEMBL Target (BACE1)
target_id = 'CHEMBL4822'
activity = new_client.activity

print("Fetching data (Limited to top 150 results)...")

# 2. Get Data - Slice [0:150] to limit download size
res = activity.filter(target_chembl_id=target_id)\
              .filter(standard_type="IC50")\
              .filter(standard_value__isnull=False)\
              .filter(standard_units='nM')[0:150]

# 3. Convert to DataFrame
df = pd.DataFrame.from_dict(res)
print(f"✅ Downloaded {len(df)} records.")

# 4. Clean Data & Calculate pIC50
df['standard_value'] = df['standard_value'].astype(float)
df['pIC50'] = -np.log10(df['standard_value'] * 1e-9)

# 5. Select Candidates (25 Active + 25 Inactive)
df = df.sort_values('pIC50', ascending=False)
df = df.drop_duplicates(subset=['canonical_smiles'])

top_active = df.head(25)
inactive = df.tail(25)
dataset = pd.concat([top_active, inactive])

# --- THE FIX IS HERE ---
# Save SMILES first, then ID. Separator is tab. No header.
dataset[['canonical_smiles', 'molecule_chembl_id']].to_csv("candidates.smi", sep='\t', index=False, header=False)
dataset.to_csv("bace1_data_with_labels.csv", index=False)

print(f"✅ Saved {len(dataset)} molecules to 'candidates.smi' (SMILES column first).")

Fetching data (Limited to top 150 results)...
✅ Downloaded 150 records.
✅ Saved 50 molecules to 'candidates.smi' (SMILES column first).


In [37]:
# 1. Create a clean directory for the ligands
!mkdir -p ligands
# 2. Convert SMILES to PDBQT and save INSIDE the directory
# Note: "ligands/ligand_.pdbqt" tells OpenBabel to put them in the folder
print("Converting molecules to 3D and saving in 'ligands/' folder...")
!obabel candidates.smi -O ligands/ligand_.pdbqt -m --gen3d -p 7.4 --partialcharge gasteiger

# 3. Verify it worked
print("✅ Conversion complete.")
!ls -1 ligands/ | head -5
print(f"Total files in folder: {len(os.listdir('ligands'))}")

Converting molecules to 3D and saving in 'ligands/' folder...
*** Open Babel Error  in Do
  3D coordinate generation failed
^C
✅ Conversion complete.
ligand_10.pdbqt
ligand_11.pdbqt
ligand_12.pdbqt
ligand_13.pdbqt
ligand_14.pdbqt
Total files in folder: 15


In [ ]:
!pip install rdkit

In [42]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# 1. Setup folders
if not os.path.exists('ligands'):
    os.makedirs('ligands')

# 2. Read the SMILES file
# We assume tab-separated: [SMILES, ID]
print("Reading candidates.smi...")
try:
    df = pd.read_csv("candidates.smi", sep='\t', names=["SMILES", "ID"])
except Exception:
    # Fallback if pandas fails to parse
    print("Pandas failed, trying manual read...")
    data = []
    with open("candidates.smi", "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                data.append({"SMILES": parts[0], "ID": parts[1]})
    df = pd.DataFrame(data)

success_count = 0
fail_count = 0

print(f"Processing {len(df)} molecules with RDKit...")

for index, row in df.iterrows():
    smi = row["SMILES"]
    mol_id = str(row["ID"])

    # Clean the ID for filename safety
    safe_id = "".join([c for c in mol_id if c.isalnum() or c in ('-','_')])

    try:
        # A. Create Molecule
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue # Skip invalid SMILES

        # B. Add Hydrogens (Crucial!)
        mol = Chem.AddHs(mol)

        # C. Generate 3D Coordinates (FIXED CODE HERE)
        # We create the parameter object first, THEN modify it
        params = AllChem.ETKDG()
        params.useRandomCoords = True # Set the flag here instead of as an argument
        params.randomSeed = 0xf00d # Optional: Fixed seed for reproducibility

        res = AllChem.EmbedMolecule(mol, params)

        # If random coords fail, try standard embedding
        if res == -1:
            res = AllChem.EmbedMolecule(mol, AllChem.ETKDG())

        if res == -1:
             # Last resort: simple distance geometry
             res = AllChem.EmbedMolecule(mol)

        if res == -1:
            raise ValueError("Could not embed molecule in 3D")

        # D. Optimize Geometry (Energy Minimization)
        try:
            AllChem.MMFFOptimizeMolecule(mol)
        except:
            pass # If force field fails, we still keep the structure

        # E. Save as PDB
        output_file = f"ligands/ligand_{index}.pdb"
        Chem.MolToPDBFile(mol, output_file)

        success_count += 1

    except Exception as e:
        print(f"⚠️ Failed on {safe_id}: {e}")
        fail_count += 1

print(f"\n✅ Done! Successfully generated {success_count} 3D structures.")
print(f"❌ Failed: {fail_count}")

Reading candidates.smi...
Processing 50 molecules with RDKit...

✅ Done! Successfully generated 50 3D structures.
❌ Failed: 0


In [44]:
print("Converting PDBs to PDBQT...")
!for f in ligands/*.pdb; do obabel "$f" -O "${f%.pdb}.pdbqt" -r --partialcharge gasteiger; done
# Delete the intermediate PDBs to save space
!rm ligands/*.pdb
print(f"✅ Conversion Complete. Total PDBQTs: {len(os.listdir('ligands'))}")

Converting PDBs to PDBQT...
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 mole

In [47]:
import glob
ligands = sorted(glob.glob("ligands/*.pdbqt"))
with open("batch_list.txt", "w") as f:
    for lig in ligands:
        f.write(lig + "\n")
print(f"✅ Updated batch_list.txt with {len(ligands)} files.")

✅ Updated batch_list.txt with 50 files.


In [48]:
cat batch_list.txt

ligands/ligand_0.pdbqt
ligands/ligand_1.pdbqt
ligands/ligand_10.pdbqt
ligands/ligand_11.pdbqt
ligands/ligand_12.pdbqt
ligands/ligand_13.pdbqt
ligands/ligand_14.pdbqt
ligands/ligand_15.pdbqt
ligands/ligand_16.pdbqt
ligands/ligand_17.pdbqt
ligands/ligand_18.pdbqt
ligands/ligand_19.pdbqt
ligands/ligand_2.pdbqt
ligands/ligand_20.pdbqt
ligands/ligand_21.pdbqt
ligands/ligand_22.pdbqt
ligands/ligand_23.pdbqt
ligands/ligand_24.pdbqt
ligands/ligand_25.pdbqt
ligands/ligand_26.pdbqt
ligands/ligand_27.pdbqt
ligands/ligand_28.pdbqt
ligands/ligand_29.pdbqt
ligands/ligand_3.pdbqt
ligands/ligand_30.pdbqt
ligands/ligand_31.pdbqt
ligands/ligand_32.pdbqt
ligands/ligand_33.pdbqt
ligands/ligand_34.pdbqt
ligands/ligand_35.pdbqt
ligands/ligand_36.pdbqt
ligands/ligand_37.pdbqt
ligands/ligand_38.pdbqt
ligands/ligand_39.pdbqt
ligands/ligand_4.pdbqt
ligands/ligand_40.pdbqt
ligands/ligand_41.pdbqt
ligands/ligand_42.pdbqt
ligands/ligand_43.pdbqt
ligands/ligand_44.pdbqt
ligands/ligand_45.pdbqt
ligands/ligand_46.pdb

In [51]:
import os
import glob
import subprocess
import time


# Get all ligand PDBQT files
ligands = sorted(glob.glob("ligands/*.pdbqt"))
total_ligands = len(ligands)


for i, ligand_file in enumerate(ligands):
    start_time = time.time()
    ligand_name = os.path.basename(ligand_file)

    print(f"----------------------------------------------------------------")
    print(f"Target {i+1}/{total_ligands}: {ligand_name}")
    print(f"----------------------------------------------------------------")


    !./autodock-gpu --ffile protein.maps.fld --lfile $ligand_file --nrun 20

    !rsync -av --update --progress /content/ "/content/drive/MyDrive/BACE1_Project/" --exclude "sample_data" --exclude ".config" --exclude "drive"



print("\n🎉 ALL JOBS COMPLETED & SYNCED!")

----------------------------------------------------------------
Target 1/50: ligand_0.pdbqt
----------------------------------------------------------------
AutoDock-GPU version: v1.6

Running 1 docking calculation

Cuda device:                              Tesla T4
Available memory on device:               14807 MB (total: 14912 MB)

CUDA Setup time 0.279221s
(Thread 1 is setting up Job #1)

Running Job #1
    Using heuristics: (capped) number of evaluations set to 11867937
             This means this docking may not be able to converge. Increasing --heurmax may improve
             convergence but will also increase runtime.
             AutoStop will not stop before 99.67% (11829247) of the set number of evaluations.
    Local-search chosen method is: ADADELTA (ad)

Rest of Setup time 0.018045s

Executing docking runs, stopping automatically after either reaching 0.15 kcal/mol standard deviation of
the best molecules of the last 4 * 5 generations, 42000 generations, or 11867937 ev

In [52]:
# KEEP THIS BLOCK AT BOTTOM
# sync to drive
!rsync -av --progress /content/ "/content/drive/MyDrive/BACE1_Project/" --exclude "sample_data" --exclude ".config" --exclude "drive"

sending incremental file list
./
ligands/

sent 3,955 bytes  received 473 bytes  8,856.00 bytes/sec
total size is 33,010,225  speedup is 7,454.88
